# 🎨 VTO Multi-Model Pipeline
## Virtual Try-On + Body Measurement — Side-by-Side Model Comparison

This notebook runs **3 virtual try-on models** on the same inputs, then uses an **AI judge** to pick the best result, and extracts **body measurements** from the person's photo.

### Pipeline Architecture
```
2 Person Images + 1 Garment Image
              ↓
  ┌───────────┼───────────┐
  │           │           │
IDM-VTON   OmniGen v1  Vertex AI
(local GPU) (fal.ai)   (Google Cloud)
  │           │           │
  └───────────┼───────────┘
              ↓
      AI Judge (Claude)
      picks best result
              ↓
     Final Output + Body Measurements
```

### Requirements
- **GPU Runtime**: Go to `Runtime → Change runtime type → T4 GPU`
- **API Keys needed**:
  - `FAL_KEY` — from [fal.ai](https://fal.ai) (for OmniGen v1)
  - `GOOGLE_CLOUD_SERVICE_ACCOUNT_JSON` — your GCP service account (for Vertex AI)
  - `ANTHROPIC_API_KEY` — from [Anthropic Console](https://console.anthropic.com) (for AI Judge)

---

## Step 0 — Install Dependencies

In [ ]:
# ============================================================
# INSTALL ALL DEPENDENCIES
# This takes 3-5 minutes on first run
# ============================================================

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate safetensors
!pip install -q mediapipe opencv-python-headless Pillow
!pip install -q fal-client anthropic google-auth google-auth-httplib2
!pip install -q huggingface_hub

# IDM-VTON specific dependencies
!pip install -q basicsr

print("\n✅ All dependencies installed!")

import torch
print(f"🔧 PyTorch: {torch.__version__}")
print(f"🖥️ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — Switch to T4 runtime!'}")
print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## Step 1 — Configuration & API Keys

Fill in your API keys below. You can also use Colab Secrets (🔑 icon in sidebar) to store them securely.

In [ ]:
import os
import json

# ============================================================
# API KEYS — Fill these in or use Colab Secrets
# ============================================================

# Try loading from Colab Secrets first, fall back to manual entry
try:
    from google.colab import userdata
    FAL_KEY = userdata.get('FAL_KEY')
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    GCP_SERVICE_ACCOUNT_JSON = userdata.get('GCP_SERVICE_ACCOUNT_JSON')
    print("✅ Loaded keys from Colab Secrets")
except:
    # ---- MANUAL ENTRY (paste your keys here) ----
    FAL_KEY = ""                        # From https://fal.ai/dashboard/keys
    ANTHROPIC_API_KEY = ""              # From https://console.anthropic.com
    GCP_SERVICE_ACCOUNT_JSON = ""       # Your GCP service account JSON string
    print("⚠️  Using manually entered keys")

os.environ['FAL_KEY'] = FAL_KEY or ""

# Google Cloud config (from your existing setup)
GCP_PROJECT_ID = "fynd-jio-impetus-non-prod"
GCP_LOCATION = "us-central1"

# Validate keys
print(f"\n📋 Key Status:")
print(f"   FAL_KEY:          {'✅ Set' if FAL_KEY else '❌ Missing — OmniGen will be skipped'}")
print(f"   ANTHROPIC_API_KEY: {'✅ Set' if ANTHROPIC_API_KEY else '❌ Missing — AI Judge will use fallback'}")
print(f"   GCP_SERVICE_ACCT: {'✅ Set' if GCP_SERVICE_ACCOUNT_JSON else '❌ Missing — Vertex AI will be skipped'}")

## Step 2 — Upload Your Images

Upload **3 images**:
1. **Person Photo 1** — Selfie / face close-up
2. **Person Photo 2** — Full body photo (head to toe)
3. **Garment Image** — The clothing item (ideally flat-lay on white/plain background)

In [ ]:
import base64
from PIL import Image
from google.colab import files
from IPython.display import display, HTML
import io

def upload_image(label):
    """Upload an image and return PIL Image + base64 string"""
    print(f"\n📤 Upload: {label}")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    img_bytes = uploaded[filename]
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    b64 = base64.b64encode(img_bytes).decode('utf-8')
    print(f"   ✅ {filename} — {img.size[0]}x{img.size[1]}")
    return img, b64, filename

def resize_for_model(img, max_size=1024):
    """Resize image keeping aspect ratio, max dimension = max_size"""
    w, h = img.size
    if max(w, h) <= max_size:
        return img
    ratio = max_size / max(w, h)
    new_w, new_h = int(w * ratio), int(h * ratio)
    return img.resize((new_w, new_h), Image.LANCZOS)

def img_to_base64(img, fmt='JPEG', quality=92):
    """Convert PIL Image to base64 data URL"""
    buf = io.BytesIO()
    img.save(buf, format=fmt, quality=quality)
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    mime = 'image/jpeg' if fmt == 'JPEG' else 'image/png'
    return f"data:{mime};base64,{b64}", b64

def save_temp_url(img, name):
    """Save image locally and return a local path (for APIs needing URLs, we'll upload to tmpfiles)"""
    path = f"/tmp/{name}.jpg"
    img.save(path, 'JPEG', quality=92)
    return path

# ---- UPLOAD ----
print("="*60)
print("UPLOAD YOUR IMAGES")
print("="*60)

person_selfie_img, person_selfie_b64, _ = upload_image("Person Photo 1 (Selfie / Face)")
person_fullbody_img, person_fullbody_b64, _ = upload_image("Person Photo 2 (Full Body)")
garment_img, garment_b64, _ = upload_image("Garment Image (Clothing Item)")

# Resize for models
person_selfie_resized = resize_for_model(person_selfie_img)
person_fullbody_resized = resize_for_model(person_fullbody_img)
garment_resized = resize_for_model(garment_img)

# Show uploaded images
print("\n" + "="*60)
print("UPLOADED IMAGES")
print("="*60)
fig, axes = __import__('matplotlib').pyplot.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, 
    [person_selfie_resized, person_fullbody_resized, garment_resized],
    ['Person: Selfie', 'Person: Full Body', 'Garment']):
    ax.imshow(img)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
__import__('matplotlib').pyplot.tight_layout()
__import__('matplotlib').pyplot.show()

---
## Step 3 — Run All 3 Models

Each model runs independently. If an API key is missing, that model is skipped.

### Model 1: IDM-VTON (Local GPU)
Best garment detail preservation. Uses full-body photo + garment image.

In [ ]:
# ============================================================
# MODEL 1: IDM-VTON — Local inference on Colab GPU
# Input: 1 person (full body) + 1 garment
# ============================================================

import time
import torch
from PIL import Image

idm_result = None
idm_time = 0

print("🔄 Model 1: IDM-VTON (Local GPU)")
print("-" * 40)

try:
    start = time.time()

    # Clone IDM-VTON repo if not already present
    if not os.path.exists('/content/IDM-VTON'):
        print("📥 Cloning IDM-VTON repository...")
        !git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON 2>/dev/null
        %cd /content/IDM-VTON
        !pip install -q -r requirements.txt 2>/dev/null
    else:
        %cd /content/IDM-VTON

    # Download model weights from HuggingFace
    from huggingface_hub import snapshot_download
    print("📥 Downloading model weights (first run only, ~5GB)...")
    model_path = snapshot_download(
        repo_id="yisol/IDM-VTON",
        local_dir="/content/idm-vton-weights",
        ignore_patterns=["*.md", "*.txt", ".git*"]
    )

    # Prepare inputs
    person_fullbody_resized.save("/tmp/idm_person.jpg", "JPEG", quality=92)
    garment_resized.save("/tmp/idm_garment.jpg", "JPEG", quality=92)

    # Run IDM-VTON inference
    # The model uses a Gradio interface internally — we call it programmatically
    import sys
    sys.path.insert(0, '/content/IDM-VTON')

    from gradio_demo.app import get_tryon_result

    print("🧠 Running inference...")
    idm_result = get_tryon_result(
        person_img=person_fullbody_resized,
        garment_img=garment_resized,
        garment_des="clothing item",
        category="upper_body",  # Change to lower_body or dresses as needed
        denoise_steps=30,
        seed=42
    )

    idm_time = time.time() - start
    print(f"\n✅ IDM-VTON complete! ({idm_time:.1f}s)")
    display(idm_result)

except Exception as e:
    print(f"\n❌ IDM-VTON failed: {e}")
    print("\n💡 Fallback: Trying via Replicate API...")
    try:
        import requests
        # Upload images to tmpfiles.org for URL access
        def upload_to_tmp(filepath):
            with open(filepath, 'rb') as f:
                r = requests.post('https://tmpfiles.org/api/v1/upload', files={'file': f})
                url = r.json()['data']['url']
                # Convert to direct download URL
                return url.replace('tmpfiles.org/', 'tmpfiles.org/dl/')

        person_fullbody_resized.save('/tmp/person_fb.jpg', 'JPEG', quality=92)
        garment_resized.save('/tmp/garment.jpg', 'JPEG', quality=92)

        person_url = upload_to_tmp('/tmp/person_fb.jpg')
        garment_url = upload_to_tmp('/tmp/garment.jpg')

        print(f"   Person URL: {person_url}")
        print(f"   Garment URL: {garment_url}")

        # Call fal.ai IDM-VTON endpoint (free!)
        import fal_client
        start = time.time()
        result = fal_client.subscribe("fal-ai/idm-vton", arguments={
            "human_image_url": person_url,
            "garment_image_url": garment_url,
            "description": "clothing item",
            "num_inference_steps": 30,
            "seed": 42
        })

        # Download result
        result_url = result['image']['url']
        img_data = requests.get(result_url).content
        idm_result = Image.open(io.BytesIO(img_data)).convert('RGB')
        idm_time = time.time() - start
        print(f"\n✅ IDM-VTON (via fal.ai) complete! ({idm_time:.1f}s)")
        display(idm_result)

    except Exception as e2:
        print(f"❌ Fallback also failed: {e2}")
        print("   IDM-VTON will be excluded from comparison.")

### Model 2: OmniGen v1 (fal.ai API)
Multi-image input — uses BOTH person photos for better identity preservation.

In [ ]:
# ============================================================
# MODEL 2: OmniGen v1 — via fal.ai API
# Input: 2 person images + 1 garment (multi-reference!)
# ============================================================

import requests
import fal_client

omnigen_result = None
omnigen_time = 0

print("🔄 Model 2: OmniGen v1 (fal.ai — Multi-Image)")
print("-" * 40)

if not FAL_KEY:
    print("⏭️  Skipped — FAL_KEY not set")
else:
    try:
        # Upload all 3 images to get URLs
        def upload_to_tmp(filepath):
            with open(filepath, 'rb') as f:
                r = requests.post('https://tmpfiles.org/api/v1/upload', files={'file': f})
                url = r.json()['data']['url']
                return url.replace('tmpfiles.org/', 'tmpfiles.org/dl/')

        person_selfie_resized.save('/tmp/omnigen_selfie.jpg', 'JPEG', quality=92)
        person_fullbody_resized.save('/tmp/omnigen_fullbody.jpg', 'JPEG', quality=92)
        garment_resized.save('/tmp/omnigen_garment.jpg', 'JPEG', quality=92)

        print("📤 Uploading images for API access...")
        selfie_url = upload_to_tmp('/tmp/omnigen_selfie.jpg')
        fullbody_url = upload_to_tmp('/tmp/omnigen_fullbody.jpg')
        garment_url_omni = upload_to_tmp('/tmp/omnigen_garment.jpg')

        print(f"   Selfie URL: {selfie_url}")
        print(f"   Full Body URL: {fullbody_url}")
        print(f"   Garment URL: {garment_url_omni}")

        # OmniGen prompt with multi-image references
        # <|image_1|> = selfie, <|image_2|> = full body, <|image_3|> = garment
        prompt = (
            "Generate a photorealistic image of the person shown in <|image_1|> and <|image_2|> "
            "wearing the clothing item shown in <|image_3|>. "
            "The person should be shown in a full-body standing pose, "
            "preserving their exact face, skin tone, hair, and body proportions. "
            "The clothing should fit naturally on their body. "
            "Studio lighting, clean background, fashion photography style."
        )

        print(f"\n📝 Prompt: {prompt[:100]}...")
        print("\n🧠 Running OmniGen v1 inference (this takes 30-60s)...")

        start = time.time()
        result = fal_client.subscribe("fal-ai/omnigen-v1", arguments={
            "prompt": prompt,
            "input_image_urls": [selfie_url, fullbody_url, garment_url_omni],
            "num_images": 1,
            "num_inference_steps": 50,
            "guidance_scale": 3.0,
            "img_guidance_scale": 1.6,
            "image_size": "square_hd",
            "seed": 42
        })

        # Download result
        result_url = result['images'][0]['url']
        img_data = requests.get(result_url).content
        omnigen_result = Image.open(io.BytesIO(img_data)).convert('RGB')
        omnigen_time = time.time() - start

        print(f"\n✅ OmniGen v1 complete! ({omnigen_time:.1f}s)")
        display(omnigen_result)

    except Exception as e:
        print(f"\n❌ OmniGen v1 failed: {e}")
        print("   OmniGen will be excluded from comparison.")

### Model 3: Google Vertex AI (virtual-try-on-001)
Your existing model. Uses full-body photo + garment image.

In [ ]:
# ============================================================
# MODEL 3: Google Vertex AI virtual-try-on-001
# Input: 1 person (full body) + 1 garment
# ============================================================

import json
import time
from google.oauth2 import service_account
from google.auth.transport.requests import Request

vertex_result = None
vertex_time = 0

print("🔄 Model 3: Google Vertex AI (virtual-try-on-001)")
print("-" * 40)

if not GCP_SERVICE_ACCOUNT_JSON:
    print("⏭️  Skipped — GCP_SERVICE_ACCOUNT_JSON not set")
else:
    try:
        start = time.time()

        # Get access token from service account
        sa_info = json.loads(GCP_SERVICE_ACCOUNT_JSON)
        credentials = service_account.Credentials.from_service_account_info(
            sa_info,
            scopes=['https://www.googleapis.com/auth/cloud-platform']
        )
        credentials.refresh(Request())
        access_token = credentials.token
        print("🔑 Google Cloud access token obtained")

        # Prepare base64 images (strip data URL prefix)
        _, person_raw_b64 = img_to_base64(person_fullbody_resized)
        _, garment_raw_b64 = img_to_base64(garment_resized)

        # Call Vertex AI
        url = f"https://{GCP_LOCATION}-aiplatform.googleapis.com/v1/projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/virtual-try-on-001:predict"

        request_body = {
            "instances": [{
                "personImage": {"image": {"bytesBase64Encoded": person_raw_b64}},
                "productImages": [{"image": {"bytesBase64Encoded": garment_raw_b64}}]
            }],
            "parameters": {
                "sampleCount": 1,
                "outputOptions": {"mimeType": "image/jpeg"}
            }
        }

        print("🧠 Calling Vertex AI...")
        response = requests.post(url,
            headers={
                'Authorization': f'Bearer {access_token}',
                'Content-Type': 'application/json'
            },
            json=request_body,
            timeout=120
        )

        if not response.ok:
            raise Exception(f"Vertex AI error {response.status_code}: {response.text[:300]}")

        data = response.json()
        result_b64 = (
            data.get('predictions', [{}])[0]
            .get('generatedImages', [{}])[0]
            .get('image', {})
            .get('bytesBase64Encoded')
        ) or (
            data.get('predictions', [{}])[0]
            .get('image', {})
            .get('bytesBase64Encoded')
        )

        if not result_b64:
            raise Exception(f"No image in response: {json.dumps(data)[:300]}")

        img_bytes = base64.b64decode(result_b64)
        vertex_result = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        vertex_time = time.time() - start

        print(f"\n✅ Vertex AI complete! ({vertex_time:.1f}s)")
        display(vertex_result)

    except Exception as e:
        print(f"\n❌ Vertex AI failed: {e}")
        print("   Vertex AI will be excluded from comparison.")

---
## Step 4 — Side-by-Side Comparison

View all results together before the AI judge picks the best one.

In [ ]:
# ============================================================
# SIDE-BY-SIDE COMPARISON
# ============================================================

import matplotlib.pyplot as plt

results = []
if idm_result:     results.append(('IDM-VTON', idm_result, idm_time))
if omnigen_result:  results.append(('OmniGen v1', omnigen_result, omnigen_time))
if vertex_result:   results.append(('Vertex AI', vertex_result, vertex_time))

if not results:
    print("❌ No models produced results. Check your API keys and try again.")
else:
    n = len(results)
    fig, axes = plt.subplots(1, n + 1, figsize=(6 * (n + 1), 8))
    if n + 1 == 2: axes = [axes[0], axes[1]]  # handle 2-subplot case

    # Show original person
    axes[0].imshow(person_fullbody_resized)
    axes[0].set_title('Original Person', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Show each model result
    for i, (name, img, t) in enumerate(results):
        axes[i + 1].imshow(img)
        axes[i + 1].set_title(f'{name}\n({t:.1f}s)', fontsize=14, fontweight='bold')
        axes[i + 1].axis('off')

    plt.suptitle('Virtual Try-On Model Comparison', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('/tmp/vto_comparison.jpg', bbox_inches='tight', dpi=150)
    plt.show()

    print(f"\n📊 Results Summary:")
    for name, _, t in results:
        print(f"   {name}: {t:.1f}s")

    # Save individual results
    for name, img, _ in results:
        safe_name = name.lower().replace(' ', '_')
        img.save(f'/tmp/result_{safe_name}.jpg', 'JPEG', quality=95)
    print(f"\n💾 Results saved to /tmp/result_*.jpg")

---
## Step 5 — AI Judge (Claude) Picks the Best Result

Claude analyzes all results and selects the winner based on:
- Face/identity preservation
- Garment fit and detail accuracy
- Overall realism and quality
- Natural body proportions

In [ ]:
# ============================================================
# AI JUDGE — Claude picks the best virtual try-on result
# ============================================================

import anthropic

winner = None
judge_reasoning = ""

print("🧑‍⚖️ AI Judge: Evaluating Results with Claude")
print("-" * 40)

if not results:
    print("⏭️  No results to judge.")
elif len(results) == 1:
    winner = results[0]
    judge_reasoning = f"Only one model produced a result: {results[0][0]}"
    print(f"\n🏆 Winner by default: {winner[0]}")
elif not ANTHROPIC_API_KEY:
    print("⏭️  ANTHROPIC_API_KEY not set — using simple heuristic instead.")
    # Fallback: pick the one with best resolution
    winner = max(results, key=lambda r: r[1].size[0] * r[1].size[1])
    judge_reasoning = f"Fallback heuristic: highest resolution ({winner[1].size[0]}x{winner[1].size[1]})"
    print(f"\n🏆 Winner (by resolution): {winner[0]}")
else:
    try:
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

        # Build message with all images
        content = []

        # Add original person photo
        _, person_b64_raw = img_to_base64(person_fullbody_resized)
        content.append({"type": "text", "text": "ORIGINAL PERSON (reference for identity matching):"})
        content.append({"type": "image", "source": {
            "type": "base64", "media_type": "image/jpeg", "data": person_b64_raw
        }})

        # Add garment
        _, garment_b64_raw = img_to_base64(garment_resized)
        content.append({"type": "text", "text": "TARGET GARMENT (what should be worn):"})
        content.append({"type": "image", "source": {
            "type": "base64", "media_type": "image/jpeg", "data": garment_b64_raw
        }})

        # Add each model result
        for i, (name, img, t) in enumerate(results):
            _, result_b64 = img_to_base64(img)
            content.append({"type": "text", "text": f"RESULT {i+1} — {name} ({t:.1f}s):"})
            content.append({"type": "image", "source": {
                "type": "base64", "media_type": "image/jpeg", "data": result_b64
            }})

        # Judge prompt
        content.append({"type": "text", "text": """
You are an expert fashion AI quality evaluator. Compare the virtual try-on results above.

Score each result (1-10) on these criteria:
1. IDENTITY PRESERVATION — Does the person look like the original? Same face, skin tone, hair?
2. GARMENT ACCURACY — Does the clothing match the target garment? Correct colors, patterns, details?
3. FIT & PROPORTIONS — Does the clothing fit naturally on the body? No distortions?
4. OVERALL REALISM — Does it look like a real photograph? No artifacts, blurriness, or uncanny valley?

Format your response as:
SCORES:
- [Model Name]: Identity X/10, Garment X/10, Fit X/10, Realism X/10 = Total X/40
- [Model Name]: ...

WINNER: [Model Name]

REASONING: [2-3 sentences explaining why the winner is best]
"""})

        print("🧠 Claude is analyzing the results...")
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1000,
            messages=[{"role": "user", "content": content}]
        )

        judge_reasoning = response.content[0].text
        print(f"\n{judge_reasoning}")

        # Parse winner from response
        for name, img, t in results:
            if name.lower() in judge_reasoning.lower().split('WINNER:')[-1][:50].lower() if 'WINNER:' in judge_reasoning else '':
                winner = (name, img, t)
                break

        if not winner:
            # Fallback: parse scores
            winner = results[0]
            print(f"\n🏆 Selected: {winner[0]}")

    except Exception as e:
        print(f"\n❌ AI Judge failed: {e}")
        winner = results[0]
        judge_reasoning = f"Fallback: first model ({results[0][0]})"

# Save winner
if winner:
    winner[1].save('/tmp/vto_winner.jpg', 'JPEG', quality=95)
    print(f"\n💾 Winner saved to /tmp/vto_winner.jpg")

---
## Step 6 — Body Measurements (MediaPipe Pose Estimation)

Extracts approximate body measurements from the full-body photo using pose keypoints.
Maps measurements to standard clothing sizes.

In [ ]:
# ============================================================
# BODY MEASUREMENTS — MediaPipe Pose Estimation
# Uses the full-body photo to estimate clothing sizes
# ============================================================

import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt

print("📏 Body Measurement Extraction")
print("-" * 40)

# Convert PIL to OpenCV
img_cv = cv2.cvtColor(np.array(person_fullbody_resized), cv2.COLOR_RGB2BGR)
img_h, img_w = img_cv.shape[:2]

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

with mp_pose.Pose(
    static_image_mode=True,
    model_complexity=2,
    min_detection_confidence=0.5
) as pose:
    results = pose.process(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))

    if not results.pose_landmarks:
        print("❌ Could not detect body pose. Ensure the full-body photo shows the person clearly.")
    else:
        landmarks = results.pose_landmarks.landmark

        # Helper: get pixel coordinates from landmark
        def get_point(landmark_id):
            lm = landmarks[landmark_id]
            return (int(lm.x * img_w), int(lm.y * img_h), lm.visibility)

        def pixel_distance(p1, p2):
            return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

        # Key landmarks (MediaPipe Pose indices)
        LEFT_SHOULDER = get_point(11)
        RIGHT_SHOULDER = get_point(12)
        LEFT_HIP = get_point(23)
        RIGHT_HIP = get_point(24)
        LEFT_ELBOW = get_point(13)
        RIGHT_ELBOW = get_point(14)
        LEFT_WRIST = get_point(15)
        RIGHT_WRIST = get_point(16)
        LEFT_KNEE = get_point(25)
        RIGHT_KNEE = get_point(26)
        LEFT_ANKLE = get_point(27)
        RIGHT_ANKLE = get_point(28)
        NOSE = get_point(0)

        # ---- Calculate pixel distances ----
        shoulder_width_px = pixel_distance(LEFT_SHOULDER, RIGHT_SHOULDER)
        hip_width_px = pixel_distance(LEFT_HIP, RIGHT_HIP)
        torso_length_px = pixel_distance(
            ((LEFT_SHOULDER[0] + RIGHT_SHOULDER[0]) // 2, (LEFT_SHOULDER[1] + RIGHT_SHOULDER[1]) // 2, 1),
            ((LEFT_HIP[0] + RIGHT_HIP[0]) // 2, (LEFT_HIP[1] + RIGHT_HIP[1]) // 2, 1)
        )
        left_arm_px = pixel_distance(LEFT_SHOULDER, LEFT_ELBOW) + pixel_distance(LEFT_ELBOW, LEFT_WRIST)
        right_arm_px = pixel_distance(RIGHT_SHOULDER, RIGHT_ELBOW) + pixel_distance(RIGHT_ELBOW, RIGHT_WRIST)
        arm_length_px = (left_arm_px + right_arm_px) / 2
        left_leg_px = pixel_distance(LEFT_HIP, LEFT_KNEE) + pixel_distance(LEFT_KNEE, LEFT_ANKLE)
        right_leg_px = pixel_distance(RIGHT_HIP, RIGHT_KNEE) + pixel_distance(RIGHT_KNEE, RIGHT_ANKLE)
        inseam_px = (left_leg_px + right_leg_px) / 2
        full_height_px = pixel_distance(NOSE, ((LEFT_ANKLE[0]+RIGHT_ANKLE[0])//2, (LEFT_ANKLE[1]+RIGHT_ANKLE[1])//2, 1))

        # ---- Estimate real-world measurements ----
        # Assumption: Average shoulder width is ~45cm for men, ~40cm for women
        # We'll ask the user for their height for better accuracy, or use a default
        print("\n💡 For accurate measurements, enter your height (or press Enter for 170cm default):")
        try:
            height_input = input("   Height in cm: ").strip()
            actual_height_cm = float(height_input) if height_input else 170.0
        except:
            actual_height_cm = 170.0

        # Scale factor: pixels to cm
        px_to_cm = actual_height_cm / full_height_px if full_height_px > 0 else 1.0

        measurements = {
            'Shoulder Width': shoulder_width_px * px_to_cm,
            'Hip Width': hip_width_px * px_to_cm,
            'Torso Length': torso_length_px * px_to_cm,
            'Arm Length': arm_length_px * px_to_cm,
            'Inseam': inseam_px * px_to_cm,
            'Chest (estimated)': shoulder_width_px * px_to_cm * 2.4,  # Rough chest = shoulder * 2.4
            'Waist (estimated)': hip_width_px * px_to_cm * 2.2,      # Rough waist from hip width
        }

        # ---- Size Mapping ----
        chest_cm = measurements['Chest (estimated)']
        waist_cm = measurements['Waist (estimated)']

        def get_top_size(chest):
            if chest < 88: return 'XS'
            elif chest < 96: return 'S'
            elif chest < 104: return 'M'
            elif chest < 112: return 'L'
            elif chest < 120: return 'XL'
            else: return 'XXL'

        def get_bottom_size(waist):
            if waist < 72: return '28'
            elif waist < 76: return '30'
            elif waist < 82: return '32'
            elif waist < 88: return '34'
            elif waist < 94: return '36'
            else: return '38+'

        top_size = get_top_size(chest_cm)
        bottom_size = get_bottom_size(waist_cm)

        # ---- Display Results ----
        print(f"\n" + "="*50)
        print(f"📏 BODY MEASUREMENTS (Height: {actual_height_cm}cm)")
        print(f"="*50)
        for name, value in measurements.items():
            print(f"   {name:.<25} {value:.1f} cm")
        print(f"\n👕 Recommended Top Size:    {top_size}")
        print(f"👖 Recommended Bottom Size: {bottom_size}")
        print(f"="*50)

        # ---- Visualize Pose ----
        annotated = img_cv.copy()
        mp_draw.draw_landmarks(
            annotated, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            mp_draw.DrawingSpec(color=(0, 255, 0), thickness=3, circle_radius=5),
            mp_draw.DrawingSpec(color=(255, 0, 0), thickness=2)
        )

        # Draw measurement lines
        cv2.line(annotated, LEFT_SHOULDER[:2], RIGHT_SHOULDER[:2], (0, 255, 255), 3)
        cv2.line(annotated, LEFT_HIP[:2], RIGHT_HIP[:2], (255, 255, 0), 3)

        # Add labels
        mid_shoulder = ((LEFT_SHOULDER[0]+RIGHT_SHOULDER[0])//2, (LEFT_SHOULDER[1]+RIGHT_SHOULDER[1])//2 - 20)
        cv2.putText(annotated, f"Shoulder: {measurements['Shoulder Width']:.0f}cm",
                    mid_shoulder, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        mid_hip = ((LEFT_HIP[0]+RIGHT_HIP[0])//2, (LEFT_HIP[1]+RIGHT_HIP[1])//2 - 20)
        cv2.putText(annotated, f"Hip: {measurements['Hip Width']:.0f}cm",
                    mid_hip, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(person_fullbody_resized)
        axes[0].set_title('Original Photo', fontsize=14)
        axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Pose + Measurements\nTop: {top_size} | Bottom: {bottom_size}', fontsize=14)
        axes[1].axis('off')
        plt.suptitle('Body Measurement Extraction', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('/tmp/body_measurements.jpg', bbox_inches='tight', dpi=150)
        plt.show()

---
## Step 7 — Final Output Summary

Everything in one view: winner image + body measurements + all comparisons.

In [ ]:
# ============================================================
# FINAL OUTPUT SUMMARY
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

print("\n" + "="*60)
print("🏆 FINAL OUTPUT SUMMARY")
print("="*60)

if winner:
    fig = plt.figure(figsize=(18, 10))
    gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

    # Row 1: Input images
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(person_selfie_resized)
    ax1.set_title('Input: Selfie', fontsize=12)
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(person_fullbody_resized)
    ax2.set_title('Input: Full Body', fontsize=12)
    ax2.axis('off')

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(garment_resized)
    ax3.set_title('Input: Garment', fontsize=12)
    ax3.axis('off')

    # Row 2: Winner (large) + measurements text
    ax4 = fig.add_subplot(gs[1, 0:2])
    ax4.imshow(winner[1])
    ax4.set_title(f'🏆 WINNER: {winner[0]} ({winner[2]:.1f}s)', fontsize=16, fontweight='bold', color='green')
    ax4.axis('off')

    # Measurements text panel
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.axis('off')
    try:
        measurement_text = f"""📏 Body Measurements
(Height: {actual_height_cm:.0f}cm)

Shoulder: {measurements['Shoulder Width']:.1f} cm
Chest:    {measurements['Chest (estimated)']:.1f} cm
Waist:    {measurements['Waist (estimated)']:.1f} cm
Hip:      {measurements['Hip Width']:.1f} cm
Arm:      {measurements['Arm Length']:.1f} cm
Torso:    {measurements['Torso Length']:.1f} cm
Inseam:   {measurements['Inseam']:.1f} cm

━━━━━━━━━━━━━━━━━━━━
👕 Top Size:    {top_size}
👖 Bottom Size: {bottom_size}"""
    except:
        measurement_text = "Body measurements\nnot available"

    ax5.text(0.1, 0.95, measurement_text, transform=ax5.transAxes,
             fontsize=13, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle('VTO Multi-Model Pipeline — Final Result', fontsize=20, fontweight='bold')
    plt.savefig('/tmp/vto_final_output.jpg', bbox_inches='tight', dpi=150)
    plt.show()

    print(f"\n💾 All outputs saved:")
    print(f"   /tmp/vto_final_output.jpg — Final summary")
    print(f"   /tmp/vto_comparison.jpg   — Model comparison")
    print(f"   /tmp/body_measurements.jpg — Pose analysis")
    print(f"   /tmp/vto_winner.jpg       — Winner image only")
    for name, _, _ in results:
        safe_name = name.lower().replace(' ', '_')
        print(f"   /tmp/result_{safe_name}.jpg — {name} result")

    # Download all outputs
    print(f"\n📥 Downloading outputs...")
    from google.colab import files
    files.download('/tmp/vto_final_output.jpg')

else:
    print("❌ No winner to display. Please ensure at least one model runs successfully.")

---
## 🔧 Troubleshooting

| Issue | Fix |
|-------|-----|
| `CUDA out of memory` | Use `Runtime → Restart runtime` then try again |
| `No GPU available` | Go to `Runtime → Change runtime type → T4 GPU` |
| `IDM-VTON clone fails` | Run `!rm -rf /content/IDM-VTON` then re-run the cell |
| `fal.ai timeout` | Increase wait time or try again (cold starts take longer) |
| `Vertex AI 403` | Check your service account has `aiplatform.endpoints.predict` permission |
| `Pose not detected` | Use a clearer full-body photo with visible limbs |

---

### Next Steps
Once you've identified the best model for your kiosk:
1. Update the Supabase edge function to use the chosen model (or multi-model with routing)
2. Add body measurement extraction to the kiosk flow
3. Map measurements to your actual catalog sizes
4. Consider prompt engineering for OmniGen to improve garment accuracy